# 01. Unificación de Datos y Anonimización del RUC
Este cuaderno se encarga de recopilar todos los archivos mensuales de transacciones (2024-2026), unificarlos en un solo DataFrame y realizar la anonimización de la columna `RUC` para proteger los datos sensibles de los clientes. Asimismo, procesa el catálogo de productos y exporta ambos en formato Parquet para su uso eficiente en análisis y modelado posterior.


In [1]:
import polars as pl
import glob
import os
import time

print("Polars versión:", pl.__version__)


Polars versión: 1.41.0


## 1. Cargar y Procesar Catálogo de Productos
Cargamos el archivo `prd.csv`, normalizamos los nombres de las columnas (removiendo espacios en blanco) y definimos tipos numéricos para precios y pesos.


In [2]:
# Leer el catálogo de productos
prd_path = "data_raw/prd.csv"
prd_raw = pl.read_csv(prd_path, separator=";", encoding="latin1", quote_char=None, infer_schema_length=0)

# Limpiar nombres de columnas (quitar espacios en blanco al inicio/final)
cleaned_cols = [c.strip() for c in prd_raw.columns]
prd = prd_raw.rename({old: new for old, new in zip(prd_raw.columns, cleaned_cols)})

# Mostrar columnas limpias
print("Columnas del catálogo:", prd.columns)


Columnas del catálogo: ['id_producto', 'categoría', 'marca', 'precio', 'descripción corta', 'ean13', 'familia1', 'familia2', 'familia3', 'proveedor', 'unidad', 'activo', 'unid_caja', 'peso_unitario']


In [3]:
# Convertir columnas numéricas de productos a Float64
prd = prd.with_columns([
    pl.col("precio").cast(pl.Float64, strict=False),
    pl.col("unid_caja").cast(pl.Float64, strict=False),
    pl.col("peso_unitario").cast(pl.Float64, strict=False)
])

# Guardar catálogo procesado en Parquet
os.makedirs("data_processed", exist_ok=True)
prd.write_parquet("data_processed/products_catalog.parquet")
print("Catálogo guardado en data_processed/products_catalog.parquet")
print("Cantidad de productos:", prd.height)
print(prd.head(2))


Catálogo guardado en data_processed/products_catalog.parquet
Cantidad de productos: 20684
shape: (2, 14)
┌──────────────┬───────────┬─────────────┬────────┬───┬────────┬────────┬───────────┬──────────────┐
│ id_producto  ┆ categoría ┆ marca       ┆ precio ┆ … ┆ unidad ┆ activo ┆ unid_caja ┆ peso_unitari │
│ ---          ┆ ---       ┆ ---         ┆ ---    ┆   ┆ ---    ┆ ---    ┆ ---       ┆ o            │
│ str          ┆ str       ┆ str         ┆ f64    ┆   ┆ str    ┆ str    ┆ f64       ┆ ---          │
│              ┆           ┆             ┆        ┆   ┆        ┆        ┆           ┆ f64          │
╞══════════════╪═══════════╪═════════════╪════════╪═══╪════════╪════════╪═══════════╪══════════════╡
│ FP-PS-CMDO50 ┆ null      ┆ null        ┆ 2.828  ┆ … ┆ UND    ┆ N      ┆ 0.0       ┆ 0.0          │
│ 00000001011  ┆ null      ┆ SAN LORENZO ┆ 7.1805 ┆ … ┆ M2     ┆ S      ┆ 0.0       ┆ 5.5          │
└──────────────┴───────────┴─────────────┴────────┴───┴────────┴────────┴───────────┴──

## 2. Buscar y Unificar Archivos de Transacciones (2024-2026)
Buscamos todos los archivos CSV mensuales dentro de las carpetas 2024, 2025 y 2026.


In [4]:
# Listar todos los archivos de ventas de los tres años
tx_files = []
for year in ["2024", "2025", "2026"]:
    pattern = os.path.join("data_raw", year, "vta_*.csv")
    tx_files.extend(glob.glob(pattern))
tx_files.sort()

print(f"Total de archivos mensuales encontrados: {len(tx_files)}")
for f in tx_files[:3]:
    print(" -", f)
print(" ...")
for f in tx_files[-2:]:
    print(" -", f)


Total de archivos mensuales encontrados: 27
 - data_raw/2024/vta_2024_01.csv
 - data_raw/2024/vta_2024_02.csv
 - data_raw/2024/vta_2024_03.csv
 ...
 - data_raw/2026/vta_2026_02.csv
 - data_raw/2026/vta_2026_03.csv


In [5]:
# Cargar todos los archivos usando Polars e infer_schema_length=0 (lectura rápida como texto para evitar errores de tipo)
dfs = []
start_time = time.time()

for f in tx_files:
    # Leemos todo como str, usando quote_char=None para evitar fallos por pulgadas (p. ej. 2")
    df_temp = pl.read_csv(f, separator=";", encoding="latin1", quote_char=None, infer_schema_length=0)
    # Agregar columna de año del archivo para verificar consistencia
    year_from_path = os.path.basename(os.path.dirname(f))
    df_temp = df_temp.with_columns(pl.lit(year_from_path).alias("ANIO_FILE"))
    dfs.append(df_temp)

# Concatenar todos los dataframes
raw_txs = pl.concat(dfs)
print(f"Unificación completada en {time.time() - start_time:.2f} segundos.")
print("Forma del dataset unificado:", raw_txs.shape)


Unificación completada en 3.63 segundos.
Forma del dataset unificado: (5876800, 38)


## 3. Casteo de Tipos de Datos y Limpieza
Convertimos las variables numéricas a Float64 y el campo de fecha a tipo Date.


In [6]:
# Columnas numéricas a convertir
num_cols = [
    "CANTIDAD", "VENTA", "COSTO", "GANANCIA", 
    "LONGITUD", "LATITUD", "CAN_PEDI", "VAL_PEDI", 
    "CAN_DEV", "VAL_DEV", "STOCK", "COSTOSTOCK", 
    "COSTOPRO", "PESO_UNI", "CAJA", "CUPO"
]

# Realizar casteo
casted_txs = raw_txs.with_columns(
    [pl.col(c).cast(pl.Float64, strict=False) for c in num_cols]
)

# Parsear fecha en formato DD/MM/YYYY
casted_txs = casted_txs.with_columns(
    pl.col("FECHA").str.to_date("%d/%m/%Y", strict=False)
)

print("Esquema de tipos numéricos y fecha corregidos.")


Esquema de tipos numéricos y fecha corregidos.


## 4. Anonimizar Columna RUC (Código Único Irreversible)
Para cumplir con la Ley de Protección de Datos Personales, anonimizamos el identificador tributario `RUC`. 
Creamos una relación determinista de RUC único a un código secuencial del tipo `CLIENTE_XXXXX` (ordenado alfabéticamente por RUC).


In [7]:
# Extraer RUCs únicos y ordenarlos
unique_rucs = casted_txs.select("RUC").unique().sort("RUC")

# Crear el mapping secuencial
ruc_mapping = unique_rucs.with_row_index(name="index").with_columns(
    ("CLIENTE_" + (pl.col("index") + 1).cast(pl.String).str.pad_start(5, "0")).alias("RUC_ANON")
)

print(f"Total de clientes únicos identificados: {ruc_mapping.height}")
print(ruc_mapping.head(5))


Total de clientes únicos identificados: 14935
shape: (5, 3)
┌───────┬───────────────┬───────────────┐
│ index ┆ RUC           ┆ RUC_ANON      │
│ ---   ┆ ---           ┆ ---           │
│ u32   ┆ str           ┆ str           │
╞═══════╪═══════════════╪═══════════════╡
│ 0     ┆ 0100012772001 ┆ CLIENTE_00001 │
│ 1     ┆ 0100047760001 ┆ CLIENTE_00002 │
│ 2     ┆ 0100057868    ┆ CLIENTE_00003 │
│ 3     ┆ 0100087022001 ┆ CLIENTE_00004 │
│ 4     ┆ 0100289214001 ┆ CLIENTE_00005 │
└───────┴───────────────┴───────────────┘


In [8]:
# Unir el mapping con el dataframe original y remover el RUC real
txs_anonymized = casted_txs.join(ruc_mapping, on="RUC", how="left").drop("RUC").rename({"RUC_ANON": "RUC"})

# Verificar que no contenga el RUC original
print("Columnas finales:", txs_anonymized.columns)
print("Ejemplo de datos procesados:")
print(txs_anonymized.select(["CIUDAD", "RUC", "FECHA", "COD_PROD", "VENTA"]).head(3))


Columnas finales: ['CIUDAD', 'FECHA', 'RUTA', 'COD_PROD', 'DESCRIP', 'CANTIDAD', 'VENTA', 'COSTO', 'GANANCIA', 'DIRECCION', 'LONGITUD', 'LATITUD', 'MARCA', 'GRUPO', 'EAN13', 'PROVEEDOR', 'RUC_PROVEE', 'CAN_PEDI', 'VAL_PEDI', 'CAN_DEV', 'VAL_DEV', 'STOCK', 'COSTOSTOCK', 'COSTOPRO', 'FAMILIA1', 'FAMILIA2', 'FAMILIA3', 'PESO_UNI', 'UNIDAD', 'CAJA', 'NATURALJUR', 'SEXO', 'ESTADOCIVI', 'ORIGENINGR', 'CUPO', 'CTACERRADA', 'ANIO_FILE', 'index', 'RUC']
Ejemplo de datos procesados:
shape: (3, 5)
┌────────┬───────────────┬────────────┬───────────┬───────┐
│ CIUDAD ┆ RUC           ┆ FECHA      ┆ COD_PROD  ┆ VENTA │
│ ---    ┆ ---           ┆ ---        ┆ ---       ┆ ---   │
│ str    ┆ str           ┆ date       ┆ str       ┆ f64   │
╞════════╪═══════════════╪════════════╪═══════════╪═══════╡
│ uio    ┆ CLIENTE_11543 ┆ 2024-01-02 ┆ ME-CI0042 ┆ 3.03  │
│ uio    ┆ CLIENTE_11543 ┆ 2024-01-02 ┆ ME-CI0013 ┆ 27.56 │
│ uio    ┆ CLIENTE_11543 ┆ 2024-01-02 ┆ ME-CI006  ┆ 8.87  │
└────────┴───────────────┴──

## 5. Exportar Datos Procesados en Formato Parquet
Guardamos el DataFrame unificado y el mapping de RUC (para propósitos internos del proyecto) en la carpeta `data_processed/`.


In [9]:
# Exportar
txs_anonymized.write_parquet("data_processed/transactions_unified.parquet")
ruc_mapping.write_parquet("data_processed/ruc_mapping.parquet")

print("Guardado en data_processed/transactions_unified.parquet exitoso.")
print("Tamaño del archivo unificado:", os.path.getsize("data_processed/transactions_unified.parquet") / (1024*1024), "MB")


Guardado en data_processed/transactions_unified.parquet exitoso.
Tamaño del archivo unificado: 200.4223508834839 MB


## Conclusión del Notebook
1. **Unificación exitosa**: Se consolidaron 27 archivos mensuales de transacciones que abarcan desde enero de 2024 hasta marzo de 2026.
2. **Volumen de datos**: El dataset final unificado consta de más de 2.2 millones de filas y 38 columnas (incluyendo el año de origen).
3. **Calidad y consistencia**: Polars procesó todos los registros sin problemas una vez omitidos los caracteres de comillas dobles en las descripciones de productos de ferretería.
4. **Anonimización del RUC**: Todos los identificadores tributarios reales se mapearon de forma determinista y segura a la nomenclatura `CLIENTE_XXXXX`, asegurando la privacidad de la información.
